In [ ]:
import torchlensmaker as tlm
import torch

class Model(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.thetaX = torch.nn.Parameter(torch.tensor(0.))
        self.thetaY = torch.nn.Parameter(torch.tensor(0.))

        self.light_source = tlm.SubChain(
            tlm.RotateX(45),
            tlm.RotateY(20),
            tlm.PointSource(15),
        )

        self.base = tlm.Sequential(
            self.light_source,
            tlm.RotateX(self.thetaX),
            tlm.RotateY(self.thetaY),
            tlm.Gap(10),
            
        )

        tlm.set_sampling3d(self.base, pupil=60)

        self.mirror = tlm.ImplicitDisk(5, solver_config=dict(num_iter=12, damping=0.9, implicit_solver="newton2"))

    def view_model(self):
        return tlm.Sequential(
            self.base,
            tlm.ReflectiveSurface(self.mirror),
        )

    def forward(self, data):
        out = self.base(tlm.SequentialData.empty(dim=3))
        coll = self.mirror(out.rays.P, out.rays.V, out.fk)
        return coll.rsm.sum()



def optimize(model, data):

    optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
    
    for i in range(150):
        optimizer.zero_grad()
        loss = model(data)
        loss.backward()
        optimizer.step()

        print(loss.item(), model.thetaX.item(), model.thetaY.item())



model = Model()

controls = {"show_optical_axis": True, "blocked_rays": "default"}
tlm.show3d(model.view_model(), end=5, controls=controls)

optimize(model, None)

tlm.show3d(model.view_model(), end=5, controls=controls)